<a href="https://colab.research.google.com/github/afafelwafi/hackai-2026/blob/main/notebooks/02_reasoning_grpo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Reasoning Models: GRPO with Verifiable Rewards on Darija Math

> HackAI 2026


Last summer, one of the most closely watched chess events wasn't Nakamura vs. Carlsen; it was the [Kaggle Arena Chess Exhibition](https://www.chess.com/events/2025-kaggle-game-arena), where frontier LLMs faced off in a bracket tournament. What made it compelling wasn't the chess itself, but what the matches revealed about reasoning under combinatorial pressure. Chess is an unforgiving stress test: every move compounds and hallucinations get punished immediately.

That's precisely why this matters beyond chess. The same reasoning failures that cause a model to hang a queen are the ones that cause it to misroute an agent, botch a multi-step tool call, or lose the thread in a long-horizon task. And it's exactly the regime where GRPO (Group Relative Policy Optimization) has been so impactful, by ditching the value-model crutch of PPO and instead scoring rollouts relative to their peers in a group, GRPO turns out to be remarkably well-suited to domains with sparse, verifiable rewards: did you win the game, did the proof check, did the code compile, did the answer match.

<img src="https://i.ibb.co/2mRrRN0/phpl7t6r4ebt0un2f4u-YKS.png" alt="phpl7t6r4ebt0un2f4u YKS" border="0">


The release of **DeepSeek-R1** in January 2025 changed the rules: a small open model trained with **GRPO** (Group Relative Policy Optimization) on **verifiable rewards** matched o1-preview on math/code. No human preference data, no reward model — just *"is the final answer correct?"*

This notebook teaches you the SOTA recipe end-to-end:

1. The math behind **GRPO** (and why it killed PPO for reasoning)
2. How to design **verifiable reward functions** for Arabic math
3. Train a small Qwen2.5 reasoning model with **TRL ≥ 0.14** in Colab
4. Compare reasoning traces in EN / AR / Darija
5. Evaluate on a held-out Darija math benchmark




## 1 · GRPO in 2 minutes

PPO (the classical RLHF algorithm) needs:
- A **value model** (1 extra network, expensive)
- A **reward model** (trained on human preferences, fragile)
- KL-tracking against the reference model (Don't bother too much, it's a measure of how two distributions are similar or not)

GRPO (DeepSeek, 2024) drops the value model entirely. The trick: for each prompt, sample $G$ completions, score them, and use the **group-relative advantage**:

$$
A_i = \frac{r_i - \text{mean}(r_{1..G})}{\text{std}(r_{1..G})}
$$

That's it. The advantage is normalized within the group, no critic required. Combined with a verifiable reward (e.g. *"does answer match ground truth?"*), it's enough to bootstrap chain-of-thought reasoning from a base model.

```
Prompt ──▶  sample G=8 completions
            │
            ▼
       reward each  →  r₁ … r₈
            │
            ▼
       A_i = (r_i - μ) / σ          (group-relative advantage)
            │
            ▼
       PPO-style clipped update on the policy, compared to reference
```

Why this matters in 2026: **every** open reasoning model (DeepSeek-R1, Qwen-QwQ, Llama-Nemotron, Magistral) uses some flavor of GRPO. It's the dominant paradigm.



In [1]:
%pip install -U -q  torchao trl math_verify

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.1/209.1 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


## 2 · Setup

In [2]:
import os, getpass, torch
if not os.getenv("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN: ")
print("CUDA:", torch.cuda.is_available(), "—", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

HF_TOKEN: ··········
CUDA: True — Tesla T4


## 3 · Load dataset [DqaDqa](https://huggingface.co/datasets/abdeljalilELmajjodi/DqaDqa)


In [3]:
from datasets import load_dataset
ds = load_dataset("abdeljalilELmajjodi/DqaDqa",split="train")
ds = ds.rename_columns({"question_darija":"question","reasoning_darija":"reasoning"})

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/2.58M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7463 [00:00<?, ? examples/s]

In [4]:
ds = ds.train_test_split(test_size=0.1, seed=42)
print(ds)

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'reasoning', 'answer'],
        num_rows: 6716
    })
    test: Dataset({
        features: ['id', 'question', 'reasoning', 'answer'],
        num_rows: 747
    })
})


## 4 · The reward function — the heart of GRPO

A reward function is a Python callable that takes a list of generated completions and returns a list of floats. The closer to "verifiable" (i.e. *programmatically checkable*), the better. We use two:

- `correctness_reward` — does the boxed answer match ground truth?
- `format_reward` — did the model emit the `<think>...</think>` and `\boxed{...}` structure?

In [5]:
# @title Reward functions
import re
from math_verify import parse, verify

THINK_RE   = re.compile(r"<think>.*?</think>", re.DOTALL)
ANSWER_RE  = re.compile(r"\\boxed\{([^}]+)\}")

def extract_answer(text: str) -> str | None:
    m = ANSWER_RE.search(text)
    return m.group(1).strip() if m else None

def correctness_reward(prompts, completions, answer, **kw):
    """Reward 1.0 if the boxed answer is numerically equal to ground truth."""
    rewards = []
    for c, gt in zip(completions, answer):
        pred = extract_answer(c[0]["content"] if isinstance(c, list) else c)
        if pred is None:
            rewards.append(0.0); continue
        try:
            ok = verify(parse(f"$\boxed{{{pred}}}$"), parse(f"$\boxed{{{gt}}}$"))
            rewards.append(1.0 if ok else 0.0)
        except Exception:
            rewards.append(1.0 if str(pred).strip() == str(gt).strip() else 0.0)
    return rewards

def format_reward(prompts, completions, **kw):
    """Reward 0.5 for proper <think> + boxed structure (independent of correctness)."""
    rewards = []
    for c in completions:
        text = c[0]["content"] if isinstance(c, list) else c
        has_think = bool(THINK_RE.search(text))
        has_box   = bool(ANSWER_RE.search(text))
        rewards.append(0.5 * (int(has_think) + int(has_box)) / 2)
    return rewards

# Quick sanity check
fake_completion = "<think>3+5=8</think>\n\\boxed{8}"
print("correct:", correctness_reward(["x"], [fake_completion], ["8"]))
print("format :", format_reward(["x"], [fake_completion]))

correct: [1.0]
format : [0.5]


## 5 · System prompt that scaffolds reasoning

In [6]:
SYSTEM_PROMPT = """You are a careful math tutor who reasons step-by-step in darija (Moroccan language) written in Arabic letters.

Your output MUST follow this exact format:

<think>
Step-by-step reasoning in Darija. Show every calculation.
</think>
\\boxed{FINAL_ANSWER}

The boxed answer must be a single number, no units, no extra text."""

def to_chat(example):
    return {
        "prompt": [
            {"role": "user", "content": SYSTEM_PROMPT + "\n\n" + example["question"]},
        ],
        "answer": example["answer"],
    }

train = ds["train"].map(to_chat)
test  = ds["test"].map(to_chat)
print(train[0])

Map:   0%|          | 0/6716 [00:00<?, ? examples/s]

Map:   0%|          | 0/747 [00:00<?, ? examples/s]

{'id': 697, 'question': 'جواد كايراجع للامتحان الوطني 2 سوايع كل ليلة، 5 د المرات فالسيمانة. وفي الويكاند (السبت والحد)، كايراجع 3 د السوايع فالنهار. إيلا كان الامتحان باقي ليه 6 د السيمانات، شحال ديال الوقت فالمجموع غادي يدوز جواد فالمراجعة؟', 'reasoning': 'جواد كايقرا 2 سوايع فـ 5 ليالي، يعني 2 * 5 = 10 د السوايع.\nوفالويكاند كايقرا 3 سوايع فالنهار (السبت والحد يعني يومين)، يعني 3 * 2 = 6 د السوايع.\nفالسيمانة وحدة، جواد كايقرا 10 + 6 = 16 ساعة.\nباقي ليه 6 سيمانات للامتحان، وهو كايقرا 16 ساعة فالسيمانة، يعني المجموع: 6 * 16 = 96 ساعة د المراجعة.', 'answer': '96', 'prompt': [{'role': 'user', 'content': 'You are a careful math tutor who reasons step-by-step in darija (Moroccan language) written in Arabic letters.\n\nYour output MUST follow this exact format:\n\n<think>\nStep-by-step reasoning in Darija. Show every calculation.\n</think>\n\\boxed{FINAL_ANSWER}\n\nThe boxed answer must be a single number, no units, no extra text.\n\nجواد كايراجع للامتحان الوطني 2 سوايع كل ليلة، 5 د المر

## 6 · GRPO training with TRL

Even on a free Colab T4 you can train a 0.5B Qwen with LoRA + GRPO. For full-finetune of 1.5–7B you'll want an A100.

In [ ]:
# @title Tiny GRPO training run (T4-friendly)
from trl import GRPOConfig, GRPOTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig

base = "MBZUAI-Paris/Atlas-Chat-2B"
tok  = AutoTokenizer.from_pretrained(base)

policy = AutoModelForCausalLM.from_pretrained(
    base, torch_dtype=torch.bfloat16, device_map="auto"
)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    task_type="CAUSAL_LM",
)

cfg = GRPOConfig(
    output_dir="grpo-darija-math",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_generations=4,           # G in the formula above
    max_completion_length=512,
    logging_steps=5,
    save_steps=200,
    bf16=True,
    report_to="wandb",            # set to "wandb" if you wire it
    beta=0.04,                   # KL coefficient
)

trainer = GRPOTrainer(
    model=policy,
    processing_class=tok,
    reward_funcs=[correctness_reward, format_reward],
    args=cfg,
    train_dataset=train.select(range(100)),  # small for the demo
    peft_config=lora,
)

# Uncomment to actually train (~15 min on T4 for 50 steps)
trainer.train()
print("Trainer ready. Uncomment trainer.train() to start.")
print(" Recommended: run on the full train set on an A100 for ~30 min.")

config.json:   0%|          | 0.00/886 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netr

## 7 · Inference with `<think>` tags

After training, your model spontaneously emits chain-of-thought between `<think>` tags. You can choose to *show* or *hide* them in the UX (DeepSeek-R1 style).

In [ ]:
test[0]

In [ ]:
# @title Generate a reasoning trace
from transformers import pipeline

# In production, swap with the trained checkpoint path
model_id = base  # "grpo-darija-math/checkpoint-XXX"
gen = pipeline("text-generation", model=model_id, torch_dtype=torch.bfloat16,
               device_map="auto", tokenizer=tok)

prompt = tok.apply_chat_template([
    {"role":"system","content": SYSTEM_PROMPT},
    {"role":"user","content": "Si Hmed 3andou 45 dirham, chra cahier b 12 w stylo b 8. Cha7al b9a 3andou?"},
], tokenize=False, add_generation_prompt=True)

out = gen(prompt, max_new_tokens=400, do_sample=False)[0]["generated_text"]
# Show only the part after the prompt
print(out[len(prompt):])

## 8 · Evaluation harness (use as your leaderboard submission)

In [ ]:
# @title Evaluate on the test split
import time

def evaluate(generate_fn, dataset, n=50):
    n = min(n, len(dataset))
    correct, t0 = 0, time.time()
    for ex in dataset.select(range(n)):
        prompt = tok.apply_chat_template(ex["prompt"], tokenize=False, add_generation_prompt=True)
        out = generate_fn(prompt)
        pred = extract_answer(out)
        if pred and str(pred).strip() == str(ex["answer"]).strip():
            correct += 1
    return {
        "accuracy": correct / n,
        "n": n,
        "seconds_per_example": (time.time() - t0) / n,
    }

def gen_fn(p):
    return gen(p, max_new_tokens=400, do_sample=False)[0]["generated_text"][len(p):]

print(evaluate(gen_fn, test, n=10))   # bump to 50 for the full eval

## 10 · Recap

You learned:
- Why **GRPO** replaced PPO as the reasoning-RL algorithm of choice
- How to design **verifiable rewards** that bootstrap CoT
- How to train a small Qwen with **TRL + LoRA** on Colab
- How to evaluate Darija/Arabic math reasoning end-to-end

> *"The biggest reasoning gain since chain-of-thought prompting."* — DeepSeek-R1 paper, on GRPO + verifiable rewards.
